# 🧪 Teste de Viabilidade — LLM Lens (GPorTuguese-2)

Este notebook testa se dá pra extrair os "internals" de um modelo pequeno em português
(tokenização, pesos de atenção, distribuição de probabilidade do próximo token)
rodando de graça no Colab, usando o **GPorTuguese-2** (GPT-2 small fine-tunado em português).

**Antes de rodar:** vá em `Ambiente de execução > Alterar tipo de ambiente de execução` e
selecione **GPU (T4)**. Isso é grátis no tier padrão do Colab.

Se tudo rodar sem erro, confirma que a abordagem é viável pro projeto completo (LLM Lens).

## 1. Instalação e checagem de GPU

In [ ]:
!pip install -q transformers torch scikit-learn matplotlib

import torch
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))
    print("VRAM total: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 2. Carregar o modelo (GPorTuguese-2)

Primeira execução baixa os pesos do Hugging Face (~500MB). Depois disso fica em cache na sessão.

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "pierreguillou/gpt2-small-portuguese"

device = "cuda" if torch.cuda.is_available() else "cpu"

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    output_attentions=True,
    output_hidden_states=True,
).to(device)
model.eval()

print(f"Modelo carregado em {time.time() - t0:.1f}s")
print(f"Parâmetros: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Camadas: {model.config.n_layer} | Cabeças de atenção por camada: {model.config.n_head}")

## 3. Teste 1 — Tokenização

In [ ]:
texto_teste = "O rato que o gato perseguiu fugiu para o porto."

ids = tokenizer.encode(texto_teste)
tokens = [tokenizer.decode([tid]) for tid in ids]

print("Texto original:", texto_teste)
print("Quantidade de tokens:", len(tokens))
print("Tokens:", tokens)

## 4. Teste 2 — Geração de texto (checar qualidade em português)

In [ ]:
prompt = "A inteligência artificial no Brasil"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

t0 = time.time()
output = model.generate(
    input_ids,
    max_new_tokens=40,
    do_sample=True,
    top_k=50,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id,
)
print(f"Gerado em {time.time() - t0:.2f}s\n")
print(tokenizer.decode(output[0], skip_special_tokens=True))

## 5. Teste 3 — Extração dos pesos de atenção (o núcleo do projeto)

In [ ]:
input_ids = tokenizer.encode(texto_teste, return_tensors="pt").to(device)

t0 = time.time()
with torch.no_grad():
    outputs = model(input_ids, output_attentions=True)
print(f"Forward pass em {time.time() - t0:.3f}s")

attentions = outputs.attentions  # tupla: (num_camadas) tensores (batch, num_cabecas, seq, seq)
print(f"Número de camadas capturadas: {len(attentions)}")
print(f"Formato de cada camada: {attentions[0].shape}  (batch, cabeças, token_origem, token_destino)")

## 6. Teste 4 — Visualizar um mapa de atenção (prova visual)

In [ ]:
import matplotlib.pyplot as plt

camada = 5   # escolha uma camada entre 0 e n_layer-1
cabeca = 0   # escolha uma cabeça entre 0 e n_head-1

attn_matrix = attentions[camada][0, cabeca].cpu().numpy()

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(attn_matrix, cmap="viridis")
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=90)
ax.set_yticklabels(tokens)
ax.set_xlabel("Token de destino (para onde olha)")
ax.set_ylabel("Token de origem (quem está olhando)")
ax.set_title(f"Atenção — Camada {camada}, Cabeça {cabeca}")
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 7. Teste 5 — Distribuição de probabilidade do próximo token

In [ ]:
import torch.nn.functional as F

prompt_parcial = "O Rio de Janeiro é uma cidade muito"
input_ids = tokenizer.encode(prompt_parcial, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(input_ids).logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)

top_probs, top_ids = torch.topk(probs, 10)

print(f"Prompt: '{prompt_parcial}'\n")
print("Top 10 próximos tokens mais prováveis:")
for prob, tid in zip(top_probs.tolist(), top_ids.tolist()):
    token_text = tokenizer.decode([tid])
    barra = "█" * int(prob * 50)
    print(f"  {token_text!r:15} {prob*100:5.1f}%  {barra}")

## 8. Converter para ONNX e publicar no Hugging Face Hub

Esta etapa prepara o modelo para rodar **direto no navegador** via Transformers.js
(arquitetura final do projeto: frontend 100% estático no GitHub Pages, sem backend).

Você vai precisar de um token de escrita do Hugging Face (Settings → Access Tokens →
New token, tipo "Write"). Cole ele quando o `notebook_login()` pedir — fica só na
sessão, não é salvo em lugar nenhum do notebook.

In [ ]:
# Nota: a partir de 2026 a Hugging Face separou o exportador de ONNX num
# pacote próprio (optimum-onnx). Instalar só 'optimum[exporters]' não é
# mais suficiente — o comando 'optimum-cli export onnx' não fica disponível.
!pip install -q "optimum-onnx[onnxruntime]" huggingface_hub

In [ ]:
# A instalação acima pode deixar o huggingface_hub em um estado inconsistente
# (mistura de arquivos de versões diferentes, causando ImportError no login).
# Esta célula força uma reinstalação limpa.
!pip install -q -U --force-reinstall "huggingface_hub[hf_xet]"
print("\n⚠️  IMPORTANTE: agora reinicie o ambiente de execução")
print("   (Ambiente de execução → Reiniciar sessão) antes de continuar,")
print("   depois pule direto para a célula do login abaixo.")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # cole seu token de escrita quando pedir

In [ ]:
# A versão do transformers pré-instalada no Colab pode ser mais nova do que
# o optimum-onnx suporta (ex: função interna get_parameter_dtype removida em
# versões recentes). Fixamos aqui a versão testada oficialmente pela
# Hugging Face para conversão ONNX.
# Após rodar esta célula, REINICIE a sessão antes de continuar.
!pip install -q "transformers[torch]==4.49.0"
print("\n⚠️  Reinicie a sessão agora (Ambiente de execução → Reiniciar sessão)")
print("   e depois pule direto para a célula do main_export abaixo.")

### Exportação manual com PyTorch puro

O sistema de configuração do `optimum` (`main_export` + `model_kwargs`) exige estender
classes internas específicas por arquitetura para declarar a atenção como saída do
grafo ONNX — o que se mostrou instável entre versões. Em vez disso, exportamos
diretamente com `torch.onnx.export`, envolvendo o modelo num wrapper que retorna a
atenção de cada camada como uma saída nomeada explícita (`attentions.0`, `attentions.1`, ...).

In [ ]:
import os
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

MODEL_NAME = "pierreguillou/gpt2-small-portuguese"
OUTPUT_DIR = "onnx_model"

base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
base_model.eval()
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)


class GPT2WithAttentions(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=True,
            use_cache=False,
        )
        return (outputs.logits,) + tuple(outputs.attentions)


wrapped = GPT2WithAttentions(base_model)

dummy = tokenizer("Isto é um texto de exemplo para exportação", return_tensors="pt")

num_layers = base_model.config.n_layer
output_names = ["logits"] + [f"attentions.{i}" for i in range(num_layers)]

dynamic_axes = {
    "input_ids": {0: "batch_size", 1: "sequence_length"},
    "attention_mask": {0: "batch_size", 1: "sequence_length"},
    "logits": {0: "batch_size", 1: "sequence_length"},
}
for name in output_names[1:]:
    dynamic_axes[name] = {0: "batch_size", 2: "sequence_length", 3: "sequence_length"}

os.makedirs(f"{OUTPUT_DIR}/onnx", exist_ok=True)

torch.onnx.export(
    wrapped,
    (dummy["input_ids"], dummy["attention_mask"]),
    f"{OUTPUT_DIR}/onnx/model.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=output_names,
    dynamic_axes=dynamic_axes,
    opset_version=14,
)

tokenizer.save_pretrained(OUTPUT_DIR)
base_model.config.save_pretrained(OUTPUT_DIR)

print("✅ Exportação concluída.")

In [ ]:
# Confirma que os arquivos foram gerados antes de seguir para a publicação
!ls -la ./onnx_model ./onnx_model/onnx

In [ ]:
# TROQUE 'seu-usuario' pelo seu nome de usuário no Hugging Face.
# Este repositório de MODELO é diferente do Space — é onde os pesos ONNX
# ficam hospedados (gratuito e sem limite para modelos públicos).
# Já exportamos direto na estrutura correta (onnx/model.onnx), então não
# precisa de nenhum passo de reorganização depois deste upload.
HF_USERNAME = "GusHil"
MODEL_REPO_ID = f"{HF_USERNAME}/gpt2-small-portuguese-onnx"

from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id=MODEL_REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path="./onnx_model",
    repo_id=MODEL_REPO_ID,
    repo_type="model",
)

print(f"\n✅ Modelo publicado em: https://huggingface.co/{MODEL_REPO_ID}")
print(f"   Use esse ID no script.js do frontend: '{MODEL_REPO_ID}'")

## 9. Resumo — VRAM usada e conclusão

In [ ]:
if torch.cuda.is_available():
    print(f"VRAM alocada: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
    print(f"VRAM reservada: {torch.cuda.memory_reserved() / 1e6:.1f} MB")
    print(f"VRAM total do T4: 16000 MB (uso está bem abaixo do limite)")

print("\n✅ Se todas as células acima rodaram sem erro, a abordagem é viável:")
print("   - Tokenização: OK")
print("   - Geração em português: OK")
print("   - Extração de atenção (todas as camadas/cabeças): OK")
print("   - Distribuição de probabilidade: OK")
print("   - Tudo rodou de graça, dentro do tier gratuito do Colab")